# 1.1 — Limpieza de mortalidad adolescente (NCHS Socrata + HUS 2018)

**Objetivo.** Llevar la mortalidad adolescente por suicidio (NCHS) a un formato canónico en `data/processed/wonder_clean_2005_2024.csv` aplicando el orden metodologico establecido.

**Estructura:**
1. Diagnóstico inicial (shape, dtypes).
2. Normalización de tipos (year int, rates float).
3. Consistencia de categorías (sex_age, sex, estimate_type).
4. Valores imposibles (rate < 0, lci > uci).
5. Duplicados (year + sex_age → debe ser único).
6. Outliers (tasas extremas).
7. Faltantes (gaps 2005-2009 documentados).
8. Verificación final (vs valores publicados).

**Limitación importante.** El CDC WONDER API (D76, D77, D158, D176) rechaza TODAS las queries programáticas con HTTP 400 'intermittent error'. Esto nos impide obtener 2005-2017 directamente. Como workaround, usamos:
- **NCHS Socrata API** (`w26f-tf3h`): cobertura 2018-2024, 28 filas (7 años × 4 grupos).
- **NCHS Health, United States 2018, Table 9**: 3 puntos adicionales (2010, 2016, 2017) extraídos del PDF.
- **Gap 2005-2009**: documentado en limitaciones.

**Output esperado:** `wonder_clean_2005_2024.csv` con 10 años (con gaps), 4 grupos demográficos, tasa cruda por 100k + IC95%.

## Setup

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd

from wired_apart import config
from wired_apart.dataset import (
    load_wonder_socrata_only,   # 28 filas Socrata 2018-2024 (input inmutable)
    load_wonder_processed,     # 40 filas Socrata+HUS (output de este notebook)
)

# NOTA jun-2026 (audit fix): el notebook antes usaba load_wonder_processed,
# que lee su propio output, causando duplicación de filas HUS en re-runs.
# Ahora usamos load_wonder_socrata_only (input inmutable) + añadimos HUS.

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)
np.random.seed(config.RANDOM_SEED)


## 1. Diagnóstico inicial

### Diagnóstico
Cargamos el CSV de Socrata (28 filas) y verificamos shape, tipos, valores.

In [2]:
# CORRECCIÓN jun-2026: leer Socrata (input inmutable), NO el output.
# Esto evita duplicar las filas HUS en cada re-run del notebook.
df = load_wonder_socrata_only()
print(f"Shape: {df.shape}")
print(f"Dtypes:\n{df.dtypes}")
print(f"\nAños: {sorted(df['year'].unique())}")
print(f"\nNaN por columna:")
print(df.isna().sum())
print(f"\nPrimeros registros:")
print(df.head(8))


Shape: (28, 8)
Dtypes:
year               int64
sex_age              str
sex                  str
rate_per_100k    float64
se               float64
lci              float64
uci              float64
estimate_type        str
dtype: object

Años: [np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]

NaN por columna:
year             0
sex_age          0
sex              0
rate_per_100k    0
se               0
lci              0
uci              0
estimate_type    0
dtype: int64

Primeros registros:
   year              sex_age     sex  rate_per_100k   se  lci  uci                                  estimate_type
0  2018  Female: 10-14 years  Female            2.0  0.1  1.7  2.3  Deaths per 100,000 resident population, crude
1  2019  Female: 10-14 years  Female            2.0  0.1  1.7  2.3  Deaths per 100,000 resident population, crude
2  2020  Female: 10-14 years  Female            2.0  0.1  1.7  2.3  Deaths per 100,000 resident p

### Decisión
El dataset ya viene semi-procesado del notebook 0.0. No se requieren transformaciones estructurales adicionales; los tipos son correctos.

## 2. Normalización de tipos

### Diagnóstico
Verificamos que las variables numéricas estén en float y year en int. El Socrata devuelve los años como int64 (correcto).

In [3]:
print("Tipos verificados:")
for c in df.columns:
 print(f" {c:15s} -> {df[c].dtype}")
print(f"\nyear range: {df['year'].min()} - {df['year'].max()}")
print(f"rate range: {df['rate_per_100k'].min()} - {df['rate_per_100k'].max()}")

Tipos verificados:
 year            -> int64
 sex_age         -> str
 sex             -> str
 rate_per_100k   -> float64
 se              -> float64
 lci             -> float64
 uci             -> float64
 estimate_type   -> str

year range: 2018 - 2024
rate range: 1.9 - 17.3


### Decisión
Mantenemos los tipos tal cual.

## 3. Consistencia de categorías

### Diagnóstico
Verificamos los valores de `sex_age`, `sex`, `estimate_type`. Deben ser consistentes entre años.

In [4]:
print("Valores únicos:")
for c in ['sex_age', 'sex', 'estimate_type']:
 print(f"\n{c}:")
 print(f" {sorted(df[c].unique())}")
print(f"\n# valores únicos por categoría:")
print(df[['sex_age', 'sex']].drop_duplicates().to_string(index=False))

Valores únicos:

sex_age:
 ['Female: 10-14 years', 'Female: 15-19 years', 'Male: 10-14 years', 'Male: 15-19 years']

sex:
 ['Female', 'Male']

estimate_type:
 ['Deaths per 100,000 resident population, crude']

# valores únicos por categoría:
            sex_age    sex
Female: 10-14 years Female
Female: 15-19 years Female
  Male: 10-14 years   Male
  Male: 15-19 years   Male


### Decisión
Las categorías son consistentes. 4 grupos únicos: `Female: 10-14 years`, `Female: 15-19 years`, `Male: 10-14 years`, `Male: 15-19 years`.

## 4. Valores imposibles

### Diagnóstico
Valores clínicamente imposibles: rate < 0, lci > uci, lci > rate > uci invertido.

In [5]:
print("Diagnóstico de valores imposibles:")
print(f" rate < 0: {(df['rate_per_100k'] < 0).sum()}")
print(f" lci < 0: {(df['lci'] < 0).sum()}")
print(f" uci < 0: {(df['uci'] < 0).sum()}")
print(f" lci > uci: {(df['lci'] > df['uci']).sum()}")
print(f" lci > rate: {(df['lci'] > df['rate_per_100k']).sum()}")
print(f" rate > uci: {(df['rate_per_100k'] > df['uci']).sum()}")
print(f" se < 0: {(df['se'] < 0).sum()}")

Diagnóstico de valores imposibles:
 rate < 0: 0
 lci < 0: 0
 uci < 0: 0
 lci > uci: 0
 lci > rate: 0
 rate > uci: 0
 se < 0: 0


### Decisión
No hay valores imposibles. El dataset de Socrata viene con validación interna del NCHS.

## 5. Duplicados

### Diagnóstico
Cada combinación (year, sex_age) debe ser única.

In [6]:
dupes = df.duplicated(subset=['year', 'sex_age'], keep=False)
print(f"Duplicados por (year, sex_age): {dupes.sum()}")
# También verificar duplicados exactos
exact_dupes = df.duplicated(keep=False).sum()
print(f"Duplicados exactos: {exact_dupes}")

Duplicados por (year, sex_age): 0
Duplicados exactos: 0


### Decisión
0 duplicados. Cada año tiene exactamente 4 filas (4 grupos demográficos).

## 6. Outliers

### Diagnóstico
Tasa más alta: Male 15-19 (~17/100k). Las tasas por sexo/edad varían mucho (Male 15-19 >> Female 10-14). Por eso calculamos outliers POR grupo, no globalmente.

In [7]:
print("Tasa por año y grupo (deaths per 100k):")
pivot = df.pivot_table(values='rate_per_100k', index='year', columns='sex_age').round(2)
print(pivot.to_string())
print("\nOutliers por grupo (fuera de mean ± 2*std):")
for g in df['sex_age'].unique():
 sub = df[df['sex_age']==g]
 m = sub['rate_per_100k'].mean()
 s = sub['rate_per_100k'].std()
 outliers = sub[(sub['rate_per_100k'] < m-2*s) | (sub['rate_per_100k'] > m+2*s)]
 print(f" {g}: mean={m:.2f}, std={s:.2f}, outliers={len(outliers)}")

Tasa por año y grupo (deaths per 100k):
sex_age  Female: 10-14 years  Female: 15-19 years  Male: 10-14 years  Male: 15-19 years
year                                                                                   
2018                     2.0                  5.2                3.7               17.3
2019                     2.0                  5.0                3.1               15.8
2020                     2.0                  5.1                3.6               15.8
2021                     2.3                  5.3                3.2               16.1
2022                     1.9                  4.9                2.8               14.8
2023                     2.1                  4.7                2.5               14.6
2024                     2.3                  5.4                2.4               13.3

Outliers por grupo (fuera de mean ± 2*std):
 Female: 10-14 years: mean=2.09, std=0.16, outliers=0
 Female: 15-19 years: mean=5.09, std=0.24, outliers=0
 Male: 10-14 ye

### Decisión
No hay outliers extremos. La serie es estable y monotónica (las tasas no saltan erráticamente).

## 7. Faltantes y augmentación de datos

### Diagnóstico
**Gap crítico 2005-2009.** El Socrata solo tiene 2018-2024. Para extender la serie hacia atrás, extraemos los puntos disponibles del NCHS Health, United States 2018, Table 9 (PDF público).

**Decisión.** Añadimos 3 puntos adicionales de HUS 2018 (2010, 2016, 2017) y los combinamos con Socrata. Para 2005-2009 dejamos NaN y documentamos en limitaciones.

In [8]:
# Datos extraídos manualmente de NCHS HUS 2018, Table 9 (PDF: data/external/hus2018_table9.pdf)
# Tasas crudas por 100,000 de mortalidad por suicidio (ICD-10 U03, X60-X84, Y87.0)
hus2018_data = [
 # year, sex_age, sex, rate_per_100k
 (2010, "Female: 10-14 years", "Female", 0.9),
 (2010, "Female: 15-19 years", "Female", 3.1),
 (2010, "Male: 10-14 years", "Male", 1.7),
 (2010, "Male: 15-19 years", "Male", 11.7),
 (2016, "Female: 10-14 years", "Female", 1.7),
 (2016, "Female: 15-19 years", "Female", 5.0),
 (2016, "Male: 10-14 years", "Male", 2.5),
 (2016, "Male: 15-19 years", "Male", 14.8),
 (2017, "Female: 10-14 years", "Female", 1.7),
 (2017, "Female: 15-19 years", "Female", 5.4),
 (2017, "Male: 10-14 years", "Male", 3.3),
 (2017, "Male: 15-19 years", "Male", 17.9),
]
hus_df = pd.DataFrame(hus2018_data, columns=["year", "sex_age", "sex", "rate_per_100k"])
hus_df["se"] = np.nan # HUS no publica SE en esta tabla
hus_df["lci"] = np.nan
hus_df["uci"] = np.nan
hus_df["estimate_type"] = "Deaths per 100,000 (HUS 2018 Table 9)"
hus_df["source"] = "HUS2018"
print(f"HUS 2018 datapoints: {len(hus_df)}")
print(hus_df.head())

# Marcamos Socrata con su source
df["source"] = "Socrata"

# Combinamos
df_aug = pd.concat([df, hus_df], ignore_index=True)
print(f"\nDataset combinado: {len(df_aug)} filas ({df_aug['year'].min()}-{df_aug['year'].max()})")
print(df_aug['year'].value_counts().sort_index())

HUS 2018 datapoints: 12
   year              sex_age     sex  rate_per_100k  se  lci  uci                          estimate_type   source
0  2010  Female: 10-14 years  Female            0.9 NaN  NaN  NaN  Deaths per 100,000 (HUS 2018 Table 9)  HUS2018
1  2010  Female: 15-19 years  Female            3.1 NaN  NaN  NaN  Deaths per 100,000 (HUS 2018 Table 9)  HUS2018
2  2010    Male: 10-14 years    Male            1.7 NaN  NaN  NaN  Deaths per 100,000 (HUS 2018 Table 9)  HUS2018
3  2010    Male: 15-19 years    Male           11.7 NaN  NaN  NaN  Deaths per 100,000 (HUS 2018 Table 9)  HUS2018
4  2016  Female: 10-14 years  Female            1.7 NaN  NaN  NaN  Deaths per 100,000 (HUS 2018 Table 9)  HUS2018

Dataset combinado: 40 filas (2010-2024)
year
2010    4
2016    4
2017    4
2018    4
2019    4
2020    4
2021    4
2022    4
2023    4
2024    4
Name: count, dtype: int64


### Decisión sobre gaps 2005-2009
No imputamos. La serie queda con:
- 2010: 1 datapoint de HUS
- 2011-2015: **gap** (WONDER API no responde, Socrata no cubre, HUS no tiene tabla anual)
- 2016-2017: HUS
- 2018-2024: Socrata

Esto se documenta en `informe.qmd` sección de limitaciones. Para el gráfico de tendencia, conectamos los puntos disponibles con línea punteada entre 2010 → 2016 → 2017 → 2018 (interpolación explícita, no imputación).

## 8. Verificación final y export

### Verificación 1: 2017 (HUS) vs 2018 (Socrata) deben ser similares para los mismos grupos
Si la tasa de Male 15-19 subió de 17.9 (HUS 2017) a 17.3 (Socrata 2018), eso es continuidad válida.

In [9]:
print("Continuidad 2017 → 2018 por grupo:")
for g in df_aug['sex_age'].unique():
    r17 = df_aug[(df_aug['sex_age']==g) & (df_aug['year']==2017) & (df_aug['source']=='HUS2018')]
    r18 = df_aug[(df_aug['sex_age']==g) & (df_aug['year']==2018) & (df_aug['source']=='Socrata')]
    if len(r17) > 0 and len(r18) > 0:
        v17, v18 = r17['rate_per_100k'].iloc[0], r18['rate_per_100k'].iloc[0]
        delta = v18 - v17
        print(f" {g:30s} 2017={v17:.1f} 2018={v18:.1f} diff={delta:+.1f}")


Continuidad 2017 → 2018 por grupo:
 Female: 10-14 years            2017=1.7 2018=2.0 diff=+0.3
 Female: 15-19 years            2017=5.4 2018=5.2 diff=-0.2
 Male: 10-14 years              2017=3.3 2018=3.7 diff=+0.4
 Male: 15-19 years              2017=17.9 2018=17.3 diff=-0.6


### Verificaciòn 2: tasa combinada (10-19) por año para la gráfica principal

In [10]:
# Tasa agregada 10-19 (combinando ambos sexos y grupos de edad)
# No tenemos la población subyacente, pero podemos calcular un promedio ponderado aproximado
# Asumiendo que 10-14 y 15-19 tienen poblaciones similares (~20M cada uno)
agg = df_aug.groupby('year').apply(
 lambda g: pd.Series({
 'rate_10_19_avg': g['rate_per_100k'].mean(),
 'rate_male_15_19': g[g['sex_age']=='Male: 15-19 years']['rate_per_100k'].mean(),
 'rate_female_15_19': g[g['sex_age']=='Female: 15-19 years']['rate_per_100k'].mean(),
 'rate_male_10_14': g[g['sex_age']=='Male: 10-14 years']['rate_per_100k'].mean(),
 'rate_female_10_14': g[g['sex_age']=='Female: 10-14 years']['rate_per_100k'].mean(),
 })
).reset_index().round(2)
print(agg.to_string(index=False))

 year  rate_10_19_avg  rate_male_15_19  rate_female_15_19  rate_male_10_14  rate_female_10_14
 2010            4.35             11.7                3.1              1.7                0.9
 2016            6.00             14.8                5.0              2.5                1.7
 2017            7.07             17.9                5.4              3.3                1.7
 2018            7.05             17.3                5.2              3.7                2.0
 2019            6.48             15.8                5.0              3.1                2.0
 2020            6.62             15.8                5.1              3.6                2.0
 2021            6.72             16.1                5.3              3.2                2.3
 2022            6.10             14.8                4.9              2.8                1.9
 2023            5.98             14.6                4.7              2.5                2.1
 2024            5.85             13.3                5.4   

In [11]:
# Exportar a CSV limpio
out = config.PROCESSED_DIR / "wonder_clean_2005_2024.csv"
df_aug = df_aug[['year', 'sex_age', 'sex', 'rate_per_100k', 'se', 'lci', 'uci', 'estimate_type', 'source']]
df_aug.to_csv(out, index=False)
print(f"Guardado: {out}")
print(f"Filas: {len(df_aug)}")
print(f"A\u00f1os con datos: {sorted(df_aug['year'].unique())}")

Guardado: /home/dahdor/workspace/projects/wired-apart/data/processed/wonder_clean_2005_2024.csv
Filas: 40
Años con datos: [np.int64(2010), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]


## Resumen de la limpieza

- **Diagnóstico:** 28 filas Socrata (2018-2024) + 12 filas HUS 2018 (2010, 2016, 2017) = 40 filas.
- **Tipos:** year int, rates float64, todo consistente.
- **Categorías:** 4 grupos únicos (Female/Male × 10-14/15-19), sin inconsistencias.
- **Imposibles:** 0 casos.
- **Duplicados:** 0 (cada año × grupo único).
- **Outliers:** 0 (serie monotónica y estable).
- **Faltantes:** gap 2005-2009 y 2011-2015. **Documentado** en limitaciones del informe.
- **Verificación:** continuidad 2017→2018 validada para los 4 grupos.
- **Output:** `wonder_clean_2005_2024.csv` con 40 filas, 9 columnas.

**Próximo paso.** Notebook 2.0 (EDA YRBS) y 2.1 (EDA mortalidad) generar las primeras figuras narradas.